# NBA Game Prediction: Preprocessing and Training Data Development

**Goal:** Prepare the data for modeling by selecting pre-game features, splitting by date, and scaling the features.

**Steps:**
1. Load the data with the engineered pre-game features
2. Select features (season to date win percentage and 10 game rolling averages for each team and its opponent) and drop early season rows with no history
3. Check for categorical features that need dummy variables
4. Split into training and testing sets by date (earlier games for training, later games for testing)
5. Standardize the numeric features (scaler fit on the training data only)
6. Save the processed data for the modeling step

**Note:** Game stats that are only known after a game ends (points, field goal percentage, plus-minus) are excluded to avoid data leakage.

In [12]:
#import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [6]:
nba = pd.read_csv('../data/nba_games_features.csv')
nba.head(3)

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,AST_R10,TOV_R10,OPP_PRIOR_WIN_PCT,OPP_PTS_R10,OPP_FG_PCT_R10,OPP_FG3_PCT_R10,OPP_DREB_R10,OPP_AST_R10,OPP_TOV_R10,DIFF_WIN_PCT
0,22023,1610612737,ATL,Atlanta Hawks,22300063,2023-10-25,ATL @ CHA,0,240,39,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,22023,1610612737,ATL,Atlanta Hawks,22300079,2023-10-27,ATL vs. NYK,0,240,42,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,22023,1610612737,ATL,Atlanta Hawks,22300097,2023-10-29,ATL @ MIL,1,240,47,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
#Select columns that I will keep
features = ['PRIOR_WIN_PCT', 'OPP_PRIOR_WIN_PCT', 'HOME_TEAM',
            'PTS_R10', 'FG_PCT_R10', 'FG3_PCT_R10', 'DREB_R10', 'AST_R10', 'TOV_R10',
            'OPP_PTS_R10', 'OPP_FG_PCT_R10', 'OPP_FG3_PCT_R10', 'OPP_DREB_R10', 'OPP_AST_R10', 'OPP_TOV_R10']
model_df = nba[['GAME_DATE','WL'] + features].dropna()
model_df = model_df.sort_values('GAME_DATE')

In [18]:
# Check for categorical features
print(X.dtypes)
# All 15 features are numeric and HOME_TEAM is already 0/1, so no dummy variables are needed.

PRIOR_WIN_PCT        float64
OPP_PRIOR_WIN_PCT    float64
HOME_TEAM              int64
PTS_R10              float64
FG_PCT_R10           float64
FG3_PCT_R10          float64
DREB_R10             float64
AST_R10              float64
TOV_R10              float64
OPP_PTS_R10          float64
OPP_FG_PCT_R10       float64
OPP_FG3_PCT_R10      float64
OPP_DREB_R10         float64
OPP_AST_R10          float64
OPP_TOV_R10          float64
dtype: object


In [9]:
#Set features and target
X = model_df[features]
y = model_df['WL']

In [10]:
#Split the data (make shuffle false so it can use prior games)
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size = 0.2,random_state=43,shuffle=False)


In [14]:
#Set scaler
SS = StandardScaler()
X_train_scaled = SS.fit_transform(X_train)
X_test_scaled = SS.transform(X_test)

In [15]:
#make df
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

In [16]:
X_train_scaled.head()

,PRIOR_WIN_PCT,OPP_PRIOR_WIN_PCT,HOME_TEAM,PTS_R10,FG_PCT_R10,FG3_PCT_R10,DREB_R10,AST_R10,TOV_R10,OPP_PTS_R10,OPP_FG_PCT_R10,OPP_FG3_PCT_R10,OPP_DREB_R10,OPP_AST_R10,OPP_TOV_R10
1153,-2.767906,-0.552994,-0.997288,-1.296436,-2.063512,-1.297639,0.058410,-1.133227,1.118498,-1.887173,-1.528082,-2.258201,-0.402597,-2.907601,1.470497
1973,-0.552931,-2.768476,1.002720,-1.886783,-1.527651,-2.258420,-0.402210,-2.907556,1.470690,-1.296768,-2.063939,-1.297448,0.058103,-1.133387,1.118305
743,1.662044,0.554748,-0.997288,-0.279727,-0.139284,-0.129495,1.071772,0.399148,0.766305,-0.509562,0.193157,0.126402,1.624484,-1.214033,0.413921
1645,0.554557,1.662489,1.002720,-0.509307,0.193600,0.126252,1.624515,-1.213878,0.414113,-0.279960,-0.139724,-0.129338,1.071643,0.398888,0.766113
5,0.554557,1.662489,-0.997288,1.360126,0.867486,-0.765408,0.795400,0.399148,0.531510,-0.935965,-1.122129,-0.661553,0.518803,-1.455971,-0.290463


In [17]:
X_train_scaled.to_csv('../data/X_train_scaled.csv', index=False)
X_test_scaled.to_csv('../data/X_test_scaled.csv', index=False)
y_train.to_csv('../data/y_train.csv', index=False)
y_test.to_csv('../data/y_test.csv', index=False)